[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/06_data-modeling.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 6 of 21*

# Data Modelling

How to design tables people can trust: grain, keys, normalisation, star schemas, facts and dimensions, slowly changing dimensions, snapshots and surrogate keys. This is what separates senior engineers.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Before designing any table, finish this sentence: one row per what? That's the grain, and it decides everything.

**What interviewers really test here:** Grain, star schemas, SCD type 2, and designing a model from a business description.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Grain: the most important word in modelling *(Basics)*
- [ ] 2. Star schemas: facts and dimensions *(Core)*
- [ ] 3. Slowly changing dimensions *(Core)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Other approaches

| Approach | Idea | Fits |
| --- | --- | --- |
| 3NF (Inmon) | normalised central warehouse | enterprise integration |
| Star (Kimball) | facts and dimensions | analytics and BI |
| Data Vault | hubs, links, satellites; full history | many changing sources, audit needs |
| One big table | everything denormalised | a simple, fast single use case |

---
# Part 2: Lessons, step by step

## Lesson 1: Grain: the most important word in modelling

*Level: Basics*

Before you design any table, finish this sentence: **"one row per \_\_\_"**. That's the **grain**.



| Table | Grain |
| --- | --- |
| orders | one row per order |
| order\_lines | one row per product in an order |
| daily\_sales | one row per store, per product, per day |



Most reporting bugs come from a misunderstood grain. If you think a table has one row per order, when it really has one per line, every count is wrong.



**Check it, don't assume it:** a grain is correct only if those columns are unique. With rows (day 1, sku 1), (day 1, sku 2), (day 2, sku 1), the smallest key is **day + sku**.



**Normalisation** stores each fact in exactly one place. If a customer's name is on every order, then changing it means updating thousands of rows, and missing one leaves "A" and "A." for one customer. That's an **update anomaly**. So split out a customers table, which is ideal for OLTP systems.



**Keys.** A **natural key** is from the business, like an email. A **surrogate key** is an integer you generate, since natural keys can change or be reused. A **hash key** is built from several columns, is deterministic, and can be computed without looking anything up.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q10](#q10), [Q11](#q11), [Q2](#q2), [Q5](#q5), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 2: Star schemas: facts and dimensions

*Level: Core*

Fully normalised data needs many joins, which makes analytics slow and hard. The **star schema**, due to Ralph Kimball, organises data for questions people ask.



```text
              dim_date
                  │
dim_customer ── fact_orders ── dim_product
                  │
              dim_store
```



|   | Fact table | Dimension table |
| --- | --- | --- |
| Holds | events, measures | descriptions |
| Example | order, qty, amount | product name, category |
| Size | huge, ever growing | small, changes slowly |
| Answers | how much? | by what? |



**Four steps to design one:** pick the business process, declare the grain, choose the dimensions, and identify the facts.



**Which measures can you add up?**



| Kind | Example | Sum across |
| --- | --- | --- |
| Additive | revenue | everything |
| Semi-additive | account balance | accounts, but not time |
| Non-additive | percentages, ratios | nothing: recompute |



Daily balances 100, 120, 90 don't total 310 of anything. The closing balance is 90.



**Date dimensions** make "weekends" or "fiscal quarter" a simple filter. **Unknown members** (key −1) mean a fact with a missing dimension isn't dropped by an inner join.



**Many-to-many** needs a bridge table, and care. An order tagged both "sale" and "gift" gets counted under each. Per-tag totals are valid, but they sum to 250, and the true revenue's 150.



**Wide table or star?** Modern column stores compress repeats well, so a wide denormalised table is often fine for simple cases. Before compression, 10M rows with a 30-byte attribute cost 300 MB wide, against about 80 MB as a star. Columnar compression narrows that gap a lot, so stars today are as much about clarity and consistent definitions as about storage.

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q6](#q6), [Q14](#q14), [Q15](#q15), [Q19](#q19), [Q16](#q16), [Q7](#q7), [Q8](#q8), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 3: Slowly changing dimensions

*Level: Core*

A customer moves from Pune to Delhi in February. Should January's sales now show Delhi? Usually not. This is the **slowly changing dimension** problem, and it's one of the most asked data engineering questions.



| Type | What it does | Use when |
| --- | --- | --- |
| **SCD 1** | overwrite the old value | history doesn't matter, like fixing a typo |
| **SCD 2** | keep every version, with dates | history matters |
| SCD 3 | a column for the previous value | only the last change matters |



**SCD 2 by hand:**



| key | id | city | valid\_from | valid\_to |
| --- | --- | --- | --- | --- |
| 1 | 100 | Pune | 2026-01-01 | **2026-02-01** |
| 2 | 100 | Delhi | 2026-02-01 | 9999-12-31 |



The update closes the old row and opens a new one. If the value hasn't changed, do nothing, which is what makes reruns safe.



**Point-in-time join:** find the version active when the event happened.



```python
JOIN dim_customer d ON d.customer_id = f.customer_id
 AND f.day >= d.valid_from AND f.day < d.valid_to
```



An order on 10 January gets Pune, and one on 10 February gets Delhi. Using half-open ranges (&gt;= start, &lt; end) means each moment matches exactly one row.



**Check integrity:** for each key, ranges should have no overlaps (they'd duplicate facts in joins) and no gaps (they'd drop facts), with exactly one current row.



The same point-in-time idea protects machine learning from leaking the future into training data.

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q18](#q18), [Q12](#q12), [Q17](#q17), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import sqlite3

SCHEMA = """
CREATE TABLE dim_customer (customer_key INT, customer_id INT, city TEXT, valid_from TEXT, valid_to TEXT);
INSERT INTO dim_customer VALUES (1,100,'Pune','2026-01-01','2026-02-01'),(2,100,'Delhi','2026-02-01','9999-12-31'),(3,200,'Goa','2026-01-01','9999-12-31');
CREATE TABLE dim_product (product_key INT, name TEXT, category TEXT);
INSERT INTO dim_product VALUES (1,'Pen','office'),(2,'Desk','office'),(3,'Tea','food');
CREATE TABLE fact_orders (order_id INT, customer_id INT, product_key INT, day TEXT, qty INT, amount INT);
INSERT INTO fact_orders VALUES (1,100,1,'2026-01-10',2,20),(2,100,3,'2026-02-10',1,5),(3,200,2,'2026-01-15',1,100),(4,100,2,'2026-03-01',1,100);
"""

def run_sql(query):
    """Run a query on a fresh database and return the rows as lists."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    rows = con.execute(query).fetchall()
    con.close()
    return [list(r) for r in rows]


<a id="q1"></a>
### Q1. Is This Column a Key?

🟢 Warm-up · *Check before trusting*

Return whether a list of rows is unique on the given columns.

**Examples**

- `rows = [{'a': 1, 'b': 1}, {'a': 1, 'b': 2}], cols = ['a']` → `False`
- `rows = [{'a': 1, 'b': 1}, {'a': 1, 'b': 2}], cols = ['a', 'b']` → `True`

In [ ]:
def is_key(rows, cols):
    """
    Return whether a list of rows is unique on the given columns.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_key, [
    (([{'a': 1, 'b': 1}, {'a': 1, 'b': 2}], ['a']), False),
    (([{'a': 1, 'b': 1}, {'a': 1, 'b': 2}], ['a', 'b']), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare every pair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build a set of key tuples.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compare every pair.

```python
def is_key(rows, cols):
    keys = [[r[c] for c in cols] for r in rows]
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            if keys[i] == keys[j]:
                return False
    return True
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Build a set of key tuples.

```python
def is_key(rows, cols):
    return len({tuple(r[c] for c in cols) for r in rows}) == len(rows)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare every pair. That costs O(n²) time and O(1) space."

"I can do better: Build a set of key tuples. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/is-this-column-a-key)

<a id="q2"></a>
### Q2. Spot the Update Anomaly

🟢 Warm-up · *Why normalise?*

Return customer ids in a flat table having more than one name, a sign that the same fact was stored inconsistently.

**Examples**

- `rows = [{'cid': 7, 'name': 'A'}, {'cid': 7, 'name': 'A.'}, {'cid': 8, 'name': 'B'}]` → `[7]`

In [ ]:
def inconsistent(rows):
    """
    Return customer ids in a flat table having more than one name, a sign that
    the same fact was stored inconsistently.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(inconsistent, [
    (([{'cid': 7, 'name': 'A'}, {'cid': 7, 'name': 'A.'}, {'cid': 8, 'name': 'B'}],), [7]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare names in nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of sets.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compare names in nested loops.

```python
def inconsistent(rows):
    out = []
    for a in rows:
        for b in rows:
            if a["cid"] == b["cid"] and a["name"] != b["name"] and a["cid"] not in out:
                out.append(a["cid"])
    return sorted(out)
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of sets.

```python
def inconsistent(rows):
    names = {}
    for r in rows:
        names.setdefault(r["cid"], set()).add(r["name"])
    return sorted(c for c, n in names.items() if len(n) > 1)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare names in nested loops. That costs O(n²) time and O(1) space."

"I can do better: A dictionary of sets. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/spot-the-update-anomaly)

<a id="q3"></a>
### Q3. Star Schema Query

🟢 Warm-up · *Fact joined to dimensions*

Return category and total fact amount, highest first.

**Examples**

- `` → `[['office', 220], ['food', 5]]`

In [ ]:
def category_sales():
    """
    Return category and total fact amount, highest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(category_sales, [
    ((), [['office', 220], ['food', 5]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up in Python after fetching.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Join the fact to the dimension, then group.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Add up in Python after fetching.

```python
def category_sales():
    totals = {}
    for cat, a in run_sql("SELECT p.category, f.amount FROM fact_orders f JOIN dim_product p ON p.product_key = f.product_key"):
        totals[cat] = totals.get(cat, 0) + a
    return sorted(([k, v] for k, v in totals.items()), key=lambda x: -x[1])
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Join the fact to the dimension, then group.

```python
def category_sales():
    return run_sql("""SELECT p.category, SUM(f.amount) AS t FROM fact_orders f
        JOIN dim_product p ON p.product_key = f.product_key GROUP BY p.category ORDER BY t DESC""")
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up in Python after fetching. That costs O(n) time and O(c) space."

"I can do better: Join the fact to the dimension, then group. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/star-schema-query)

<a id="q4"></a>
### Q4. SCD Type 1: Overwrite

🟢 Warm-up · *No history kept*

Apply updates to a dimension keyed by id, overwriting values and adding new ids. Return rows sorted by id.

**Examples**

- `dim = [{'id': 1, 'city': 'Pune'}, {'id': 2, 'city': 'Goa'}], updates = [{'id': 1, 'city': 'Delhi'}, {'id': 3, 'city': 'Agra'}]` → `[{'id': 1, 'city': 'Delhi'}, {'id': 2, 'city': 'Goa'}, {'id': 3, 'city': 'Agra'}]`

In [ ]:
def scd1(dim, updates):
    """
    Apply updates to a dimension keyed by id, overwriting values and adding new
    ids. Return rows sorted by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(scd1, [
    (([{'id': 1, 'city': 'Pune'}, {'id': 2, 'city': 'Goa'}], [{'id': 1, 'city': 'Delhi'}, {'id': 3, 'city': 'Agra'}]), [{'id': 1, 'city': 'Delhi'}, {'id': 2, 'city': 'Goa'}, {'id': 3, 'city': 'Agra'}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search the list for each update.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary keyed by id.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Search the list for each update.

```python
def scd1(dim, updates):
    out = [dict(r) for r in dim]
    for u in updates:
        for r in out:
            if r["id"] == u["id"]:
                r.update(u)
                break
        else:
            out.append(dict(u))
    return sorted(out, key=lambda r: r["id"])
```

**Cost:** O(n·u) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary keyed by id.

```python
def scd1(dim, updates):
    merged = {r["id"]: r for r in dim}
    merged.update({u["id"]: u for u in updates})
    return [merged[k] for k in sorted(merged)]
```

**Cost:** O(n + u) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search the list for each update. That costs O(n·u) time and O(n) space."

"I can do better: A dictionary keyed by id. That's O(n + u) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/scd-type-1-overwrite)

<a id="q5"></a>
### Q5. Surrogate Keys

🟢 Warm-up · *Stable internal ids*

Assign integer keys from 1 to natural keys in first-seen order. Return the mapping and the rows as keys.

**Examples**

- `values = ['u9', 'u3', 'u9']` → `({'u9': 1, 'u3': 2}, [1, 2, 1])`

In [ ]:
def surrogate(values):
    """
    Assign integer keys from 1 to natural keys in first-seen order. Return the
    mapping and the rows as keys.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(surrogate, [
    ((['u9', 'u3', 'u9'],), ({'u9': 1, 'u3': 2}, [1, 2, 1])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list with `index`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary with `setdefault`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A list with `index`.

```python
def surrogate(values):
    seen = []
    keys = []
    for v in values:
        if v not in seen:
            seen.append(v)
        keys.append(seen.index(v) + 1)
    return ({v: i + 1 for i, v in enumerate(seen)}, keys)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary with `setdefault`.

```python
def surrogate(values):
    mapping = {}
    keys = [mapping.setdefault(v, len(mapping) + 1) for v in values]
    return (mapping, keys)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list with `index`. That costs O(n²) time and O(n) space."

"I can do better: A dictionary with `setdefault`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/surrogate-keys)

<a id="q6"></a>
### Q6. Build a Date Dimension

🟢 Warm-up · *Every warehouse has one*

For each date string, return key (YYYYMMDD as int), year, month, weekday name, and whether it's a weekend.

**Examples**

- `days = ['2026-01-03', '2026-01-05']` → `[{'key': 20260103, 'year': 2026, 'month': 1, 'weekday': 'Saturday', 'weekend': True}, {'key': 20260105, 'year': 2026, 'month': 1, 'weekday': 'Monday', 'weekend': False}]`

In [ ]:
def date_dim(days):
    """
    For each date string, return key (YYYYMMDD as int), year, month, weekday
    name, and whether it's a weekend.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(date_dim, [
    ((['2026-01-03', '2026-01-05'],), [{'key': 20260103, 'year': 2026, 'month': 1, 'weekday': 'Saturday', 'weekend': True}, {'key': 20260105, 'year': 2026, 'month': 1, 'weekday': 'Monday', 'weekend': False}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Parse parts by hand and use the datetime weekday.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`date.fromisoformat` and `strftime`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Parse parts by hand and use the datetime weekday.

```python
def date_dim(days):
    from datetime import date
    names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    out = []
    for d in days:
        y, m, dd = map(int, d.split("-"))
        wd = date(y, m, dd).weekday()
        out.append({"key": y * 10000 + m * 100 + dd, "year": y, "month": m, "weekday": names[wd], "weekend": wd >= 5})
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `date.fromisoformat` and `strftime`.

```python
def date_dim(days):
    from datetime import date
    parsed = [date.fromisoformat(d) for d in days]
    return [{"key": int(d.strftime("%Y%m%d")), "year": d.year, "month": d.month, "weekday": d.strftime("%A"), "weekend": d.weekday() >= 5} for d in parsed]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Parse parts by hand and use the datetime weekday. That costs O(n) time and O(n) space."

"I can do better: `date.fromisoformat` and `strftime`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/build-a-date-dimension)

<a id="q7"></a>
### Q7. Wide Table or Star?

🟢 Warm-up · *Storage trade-offs*

A fact has n rows. Repeating a text attribute of s bytes on each row costs n × s. A dimension stores d distinct values and an 8-byte key per fact. Return (wide MB, star MB) with MB = 10⁶ bytes, before compression.

**Examples**

- `n = 10000000, s = 30, d = 1000` → `(300.0, 80.038)`

In [ ]:
def wide_vs_star(n, s, d):
    """
    A fact has n rows. Repeating a text attribute of s bytes on each row costs n
    × s. A dimension stores d distinct values and an 8-byte key per fact. Return
    (wide MB, star MB) with MB = 10⁶ bytes, before compression.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(wide_vs_star, [
    ((10000000, 30, 1000), (300.0, 80.038)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute each part.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two formulas.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute each part.

```python
def wide_vs_star(n, s, d):
    wide = n * s
    keys = n * 8
    dimension = d * (s + 8)
    return (wide / 1e6, (keys + dimension) / 1e6)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two formulas.

```python
def wide_vs_star(n, s, d):
    return (n * s / 1e6, (n * 8 + d * (s + 8)) / 1e6)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute each part. That costs O(1) time and O(1) space."

"I can do better: Two formulas. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/wide-table-or-star)

<a id="q8"></a>
### Q8. Check Foreign Keys

🟢 Warm-up · *Referential integrity*

Warehouses often don't enforce foreign keys, so you check. Return fact keys missing from the dimension, sorted and unique.

**Examples**

- `fact_keys = [1, 2, 2, 9, 7], dim_keys = [1, 2, 3]` → `[7, 9]`

In [ ]:
def orphan_keys(fact_keys, dim_keys):
    """
    Warehouses often don't enforce foreign keys, so you check. Return fact keys
    missing from the dimension, sorted and unique.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(orphan_keys, [
    (([1, 2, 2, 9, 7], [1, 2, 3]), [7, 9]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search for each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set difference.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Search for each.

```python
def orphan_keys(fact_keys, dim_keys):
    out = []
    for k in fact_keys:
        if k not in dim_keys and k not in out:
            out.append(k)
    return sorted(out)
```

**Cost:** O(f·d) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set difference.

```python
def orphan_keys(fact_keys, dim_keys):
    return sorted(set(fact_keys) - set(dim_keys))
```

**Cost:** O(f + d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search for each. That costs O(f·d) time and O(f) space."

"I can do better: Set difference. That's O(f + d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/check-foreign-keys)

<a id="q9"></a>
### Q9. Rollup Must Reconcile

🟢 Warm-up · *Trust but verify*

Daily totals must add up to the fact table's total per month. Return months where they don't, sorted. Rows are (month, amount).

**Examples**

- `facts = [('01', 10), ('01', 5), ('02', 7)], rollup = {'01': 15, '02': 8}` → `['02']`

In [ ]:
def bad_months(facts, rollup):
    """
    Daily totals must add up to the fact table's total per month. Return months
    where they don't, sorted. Rows are (month, amount).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bad_months, [
    (([('01', 10), ('01', 5), ('02', 7)], {'01': 15, '02': 8}), ['02']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up in a loop, and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `defaultdict`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Add up in a loop, and compare.

```python
def bad_months(facts, rollup):
    totals = {}
    for m, a in facts:
        totals[m] = totals.get(m, 0) + a
    return sorted(m for m in rollup if totals.get(m, 0) != rollup[m])
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A `defaultdict`.

```python
def bad_months(facts, rollup):
    from collections import defaultdict
    totals = defaultdict(int)
    for m, a in facts:
        totals[m] += a
    return sorted(m for m, v in rollup.items() if totals[m] != v)
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up in a loop, and compare. That costs O(n) time and O(m) space."

"I can do better: A `defaultdict`. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/rollup-must-reconcile)

<a id="q10"></a>
### Q10. Find the Grain

🟡 Interview standard · *One row per what?*

Return the smallest list of columns (fewest first, then in the given order) that uniquely identifies rows.

**Examples**

- `rows = [{'day': 1, 'store': 'a', 'sku': 1}, {'day': 1, 'store': 'a', 'sku': 2}, {'day': 2, 'store': 'a', 'sku': 1}], cols = ['day', 'store', 'sku']` → `['day', 'sku']`

In [ ]:
def find_grain(rows, cols):
    """
    Return the smallest list of columns (fewest first, then in the given order)
    that uniquely identifies rows.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_grain, [
    (([{'day': 1, 'store': 'a', 'sku': 1}, {'day': 1, 'store': 'a', 'sku': 2}, {'day': 2, 'store': 'a', 'sku': 1}], ['day', 'store', 'sku']), ['day', 'sku']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try combinations by hand, with nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.combinations` by increasing size.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Try combinations by hand, with nested loops.

```python
def find_grain(rows, cols):
    subsets = [[]]
    for c in cols:
        subsets += [s + [c] for s in subsets]
    subsets = sorted(subsets[1:], key=len)
    for s in subsets:
        seen = {tuple(r[c] for c in s) for r in rows}
        if len(seen) == len(rows):
            return s
    return None
```

**Cost:** O(2ᶜ·n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.combinations` by increasing size.

```python
def find_grain(rows, cols):
    import itertools
    for size in range(1, len(cols) + 1):
        for combo in itertools.combinations(cols, size):
            if len({tuple(r[c] for c in combo) for r in rows}) == len(rows):
                return list(combo)
    return None
```

**Cost:** O(2ᶜ·n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try combinations by hand, with nested loops. That costs O(2ᶜ·n) time and O(n) space."

"I can do better: `itertools.combinations` by increasing size. That's O(2ᶜ·n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/find-the-grain)

<a id="q11"></a>
### Q11. Normalise a Flat Table

🟡 Interview standard · *Each fact in one place*

Rows repeat customer details. Split into (customers, orders): customers is unique by cid with name; orders keeps oid and cid.

**Examples**

- `rows = [{'oid': 1, 'cid': 7, 'name': 'A'}, {'oid': 2, 'cid': 7, 'name': 'A'}, {'oid': 3, 'cid': 8, 'name': 'B'}]` → `([{'cid': 7, 'name': 'A'}, {'cid': 8, 'name': 'B'}], [{'oid': 1, 'cid': 7}, {'oid': 2, 'cid': 7}, {'oid': 3, 'cid': 8}])`

In [ ]:
def normalise(rows):
    """
    Rows repeat customer details. Split into (customers, orders): customers is
    unique by cid with name; orders keeps oid and cid.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(normalise, [
    (([{'oid': 1, 'cid': 7, 'name': 'A'}, {'oid': 2, 'cid': 7, 'name': 'A'}, {'oid': 3, 'cid': 8, 'name': 'B'}],), ([{'cid': 7, 'name': 'A'}, {'cid': 8, 'name': 'B'}], [{'oid': 1, 'cid': 7}, {'oid': 2, 'cid': 7}, {'oid': 3, 'cid': 8}])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and check whether each customer's been seen.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary keyed by cid.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loop and check whether each customer's been seen.

```python
def normalise(rows):
    customers, orders = [], []
    for r in rows:
        if not any(c["cid"] == r["cid"] for c in customers):
            customers.append({"cid": r["cid"], "name": r["name"]})
        orders.append({"oid": r["oid"], "cid": r["cid"]})
    return (customers, orders)
```

**Cost:** O(n·c) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary keyed by cid.

```python
def normalise(rows):
    customers = {r["cid"]: {"cid": r["cid"], "name": r["name"]} for r in rows}
    return (list(customers.values()), [{"oid": r["oid"], "cid": r["cid"]} for r in rows])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and check whether each customer's been seen. That costs O(n·c) time and O(n) space."

"I can do better: A dictionary keyed by cid. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/normalise-a-flat-table)

<a id="q12"></a>
### Q12. Check SCD 2 Integrity

🟡 Interview standard · *No gaps or overlaps*

Rows are (start, end) for one key, with end None for current. Return problems: 'overlap', 'gap' or 'many current', in that order, for those found.

**Examples**

- `ranges = [(1, 5), (5, None)]` → `[]`
- `ranges = [(1, 5), (4, 8), (9, None), (10, None)]` → `['overlap', 'gap', 'many current']`

In [ ]:
def scd_integrity(ranges):
    """
    Rows are (start, end) for one key, with end None for current. Return
    problems: 'overlap', 'gap' or 'many current', in that order, for those
    found.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(scd_integrity, [
    (([(1, 5), (5, None)],), []),
    (([(1, 5), (4, 8), (9, None), (10, None)],), ['overlap', 'gap', 'many current']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare each neighbour after sorting, with separate flags.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same check, via zip on neighbours.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compare each neighbour after sorting, with separate flags.

```python
def scd_integrity(ranges):
    r = sorted(ranges, key=lambda x: x[0])
    overlap = gap = False
    for i in range(len(r) - 1):
        end = r[i][1]
        if end is None or end > r[i + 1][0]:
            overlap = True
        elif end < r[i + 1][0]:
            gap = True
    many = len([x for x in r if x[1] is None]) > 1
    out = []
    if overlap:
        out.append("overlap")
    if gap:
        out.append("gap")
    if many:
        out.append("many current")
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same check, via zip on neighbours.

```python
def scd_integrity(ranges):
    r = sorted(ranges, key=lambda x: x[0])
    pairs = list(zip(r, r[1:]))
    flags = [("overlap", any(a[1] is None or a[1] > b[0] for a, b in pairs)),
             ("gap", any(a[1] is not None and a[1] < b[0] for a, b in pairs)),
             ("many current", sum(x[1] is None for x in r) > 1)]
    return [name for name, hit in flags if hit]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare each neighbour after sorting, with separate flags. That costs O(n log n) time and O(n) space."

"I can do better: Same check, via zip on neighbours. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/check-scd-2-integrity)

<a id="q13"></a>
### Q13. Hash Key From Several Columns

🟡 Interview standard · *Deterministic keys*

Build a key with md5 of the values joined by '|', treating None as an empty string, and keep the first 8 hex characters.

**Examples**

- `values = [1, 'Pune', None]` → `'6461e899'`

In [ ]:
def hash_key(values):
    """
    Build a key with md5 of the values joined by '|', treating None as an empty
    string, and keep the first 8 hex characters.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hash_key, [
    (([1, 'Pune', None],), '6461e899'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the text in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`join` with a generator.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Build the text in a loop.

```python
def hash_key(values):
    import hashlib
    text = ""
    for i, v in enumerate(values):
        if i:
            text += "|"
        text += "" if v is None else str(v)
    return hashlib.md5(text.encode()).hexdigest()[:8]
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `join` with a generator.

```python
def hash_key(values):
    import hashlib
    return hashlib.md5("|".join("" if v is None else str(v) for v in values).encode()).hexdigest()[:8]
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the text in a loop. That costs O(k) time and O(k) space."

"I can do better: `join` with a generator. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/hash-key-from-several-columns)

<a id="q14"></a>
### Q14. Additive or Not?

🟡 Interview standard · *Which measures can you SUM?*

Given daily balance snapshots (a semi-additive measure), return (the wrong sum across days, the correct closing balance at the last day).

**Examples**

- `rows = [('d1', 100), ('d2', 120), ('d3', 90)]` → `(310, 90)`

In [ ]:
def closing_balance(rows):
    """
    Given daily balance snapshots (a semi-additive measure), return (the wrong
    sum across days, the correct closing balance at the last day).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(closing_balance, [
    (([('d1', 100), ('d2', 120), ('d3', 90)],), (310, 90)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop tracking the sum and the last day.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sum` and `max`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop tracking the sum and the last day.

```python
def closing_balance(rows):
    total = 0
    last_day, last_value = None, None
    for day, value in rows:
        total += value
        if last_day is None or day > last_day:
            last_day, last_value = day, value
    return (total, last_value)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `sum` and `max`.

```python
def closing_balance(rows):
    return (sum(v for _, v in rows), max(rows)[1])
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop tracking the sum and the last day. That costs O(n) time and O(1) space."

"I can do better: `sum` and `max`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/additive-or-not)

<a id="q15"></a>
### Q15. Daily Snapshot From Events

🟡 Interview standard · *Accumulate to a state*

Given deposits and withdrawals as (day, delta), return the balance at the end of each day from first to last, including days with no events.

**Examples**

- `events = [(1, 100), (3, -30), (3, 10), (5, 5)]` → `[[1, 100], [2, 100], [3, 80], [4, 80], [5, 85]]`

In [ ]:
def daily_balance(events):
    """
    Given deposits and withdrawals as (day, delta), return the balance at the
    end of each day from first to last, including days with no events.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(daily_balance, [
    (([(1, 100), (3, -30), (3, 10), (5, 5)],), [[1, 100], [2, 100], [3, 80], [4, 80], [5, 85]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each day, sum all events up to it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary per day, then a running sum.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** For each day, sum all events up to it.

```python
def daily_balance(events):
    days = [e[0] for e in events]
    return [[d, sum(v for day, v in events if day <= d)] for d in range(min(days), max(days) + 1)]
```

**Cost:** O(d·n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary per day, then a running sum.

```python
def daily_balance(events):
    import itertools
    per_day = {}
    for d, v in events:
        per_day[d] = per_day.get(d, 0) + v
    lo, hi = min(per_day), max(per_day)
    running = itertools.accumulate(per_day.get(d, 0) for d in range(lo, hi + 1))
    return [[d, b] for d, b in zip(range(lo, hi + 1), running)]
```

**Cost:** O(d + n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each day, sum all events up to it. That costs O(d·n) time and O(d) space."

"I can do better: A dictionary per day, then a running sum. That's O(d + n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/daily-snapshot-from-events)

<a id="q16"></a>
### Q16. Unknown Dimension Member

🟡 Interview standard · *Don't drop facts*

Facts refer to dimension keys. Replace missing keys with −1 ('unknown'), so an inner join won't lose them. Return (facts with fixed keys, count fixed).

**Examples**

- `facts = [{'id': 1, 'k': 10}, {'id': 2, 'k': 99}], keys = [10, 20]` → `([{'id': 1, 'k': 10}, {'id': 2, 'k': -1}], 1)`

In [ ]:
def unknown_member(facts, keys):
    """
    Facts refer to dimension keys. Replace missing keys with −1 ('unknown'), so
    an inner join won't lose them. Return (facts with fixed keys, count fixed).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unknown_member, [
    (([{'id': 1, 'k': 10}, {'id': 2, 'k': 99}], [10, 20]), ([{'id': 1, 'k': 10}, {'id': 2, 'k': -1}], 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search the list for each fact.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set for membership.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Search the list for each fact.

```python
def unknown_member(facts, keys):
    out, fixed = [], 0
    for f in facts:
        if f["k"] in keys:
            out.append(dict(f))
        else:
            out.append({**f, "k": -1})
            fixed += 1
    return (out, fixed)
```

**Cost:** O(f·d) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set for membership.

```python
def unknown_member(facts, keys):
    known = set(keys)
    out = [{**f, "k": f["k"] if f["k"] in known else -1} for f in facts]
    return (out, sum(f["k"] == -1 for f in out))
```

**Cost:** O(f + d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search the list for each fact. That costs O(f·d) time and O(f) space."

"I can do better: A set for membership. That's O(f + d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/unknown-dimension-member)

<a id="q17"></a>
### Q17. Point-in-Time Join

🔴 Stretch · *SCD 2: what was true then?*

Join each order to the customer's city as of the order date, using valid\_from and valid\_to. Return order\_id and city by order\_id.

**Examples**

- `` → `[[1, 'Pune'], [2, 'Delhi'], [3, 'Goa'], [4, 'Delhi']]`

In [ ]:
def city_at_order():
    """
    Join each order to the customer's city as of the order date, using
    valid_from and valid_to. Return order_id and city by order_id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(city_at_order, [
    ((), [[1, 'Pune'], [2, 'Delhi'], [3, 'Goa'], [4, 'Delhi']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A correlated subquery per order.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A range condition in the join: valid\_from ≤ day &lt; valid\_to.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A correlated subquery per order.

```python
def city_at_order():
    return run_sql("""SELECT order_id, (SELECT city FROM dim_customer d WHERE d.customer_id = f.customer_id
        AND d.valid_from <= f.day AND f.day < d.valid_to) FROM fact_orders f ORDER BY order_id""")
```

**Cost:** O(n·m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A range condition in the join: valid\_from ≤ day &lt; valid\_to.

```python
def city_at_order():
    return run_sql("""SELECT f.order_id, d.city FROM fact_orders f JOIN dim_customer d
        ON d.customer_id = f.customer_id AND f.day >= d.valid_from AND f.day < d.valid_to ORDER BY f.order_id""")
```

**Cost:** O(n·m) without indexes time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A correlated subquery per order. That costs O(n·m) time and O(n) space."

"I can do better: A range condition in the join: valid\_from ≤ day &lt; valid\_to. That's O(n·m) without indexes time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/point-in-time-join)

<a id="q18"></a>
### Q18. SCD Type 2: Keep History

🔴 Stretch · *Close the old, open the new*

dim rows have id, city, start, end (None = current). For each update (id, city) on day: if the city changed, close the current row and add a new one; new ids get a new row. Return rows sorted by id, start.

**Examples**

- `dim = [{'id': 1, 'city': 'Pune', 'start': 1, 'end': None}], updates = [{'id': 1, 'city': 'Delhi'}, {'id': 2, 'city': 'Goa'}], day = 5` → `[{'id': 1, 'city': 'Pune', 'start': 1, 'end': 5}, {'id': 1, 'city': 'Delhi', 'start': 5, 'end': None}, {'id': 2, 'city': 'Goa', 'start': 5, 'end': None}]`
- `dim = [{'id': 1, 'city': 'Pune', 'start': 1, 'end': None}], updates = [{'id': 1, 'city': 'Pune'}], day = 5` → `[{'id': 1, 'city': 'Pune', 'start': 1, 'end': None}]`

In [ ]:
def scd2(dim, updates, day):
    """
    dim rows have id, city, start, end (None = current). For each update (id,
    city) on day: if the city changed, close the current row and add a new one;
    new ids get a new row. Return rows sorted by id, start.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(scd2, [
    (([{'id': 1, 'city': 'Pune', 'start': 1, 'end': None}], [{'id': 1, 'city': 'Delhi'}, {'id': 2, 'city': 'Goa'}], 5), [{'id': 1, 'city': 'Pune', 'start': 1, 'end': 5}, {'id': 1, 'city': 'Delhi', 'start': 5, 'end': None}, {'id': 2, 'city': 'Goa', 'start': 5, 'end': None}]),
    (([{'id': 1, 'city': 'Pune', 'start': 1, 'end': None}], [{'id': 1, 'city': 'Pune'}], 5), [{'id': 1, 'city': 'Pune', 'start': 1, 'end': None}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search for the current row for each update.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of current rows.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Search for the current row for each update.

```python
def scd2(dim, updates, day):
    out = [dict(r) for r in dim]
    for u in updates:
        current = None
        for r in out:
            if r["id"] == u["id"] and r["end"] is None:
                current = r
        if current is None:
            out.append({"id": u["id"], "city": u["city"], "start": day, "end": None})
        elif current["city"] != u["city"]:
            current["end"] = day
            out.append({"id": u["id"], "city": u["city"], "start": day, "end": None})
    return sorted(out, key=lambda r: (r["id"], r["start"]))
```

**Cost:** O(n·u) time, O(n + u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of current rows.

```python
def scd2(dim, updates, day):
    out = [dict(r) for r in dim]
    current = {r["id"]: r for r in out if r["end"] is None}
    for u in updates:
        row = current.get(u["id"])
        if row is not None and row["city"] == u["city"]:
            continue
        if row is not None:
            row["end"] = day
        new = {"id": u["id"], "city": u["city"], "start": day, "end": None}
        out.append(new)
        current[u["id"]] = new
    return sorted(out, key=lambda r: (r["id"], r["start"]))
```

**Cost:** O(n + u) time, O(n + u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search for the current row for each update. That costs O(n·u) time and O(n + u) space."

"I can do better: A dictionary of current rows. That's O(n + u) time and O(n + u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/scd-type-2-keep-history)

<a id="q19"></a>
### Q19. Bridge Table: Avoid Double Counting

🔴 Stretch · *Many-to-many*

An order can have several tags. Return (revenue by naive join, true revenue, revenue per tag).

**Examples**

- `orders = [(1, 100), (2, 50)], tags = [(1, 'sale'), (1, 'gift'), (2, 'sale')]` → `(250, 150, {'gift': 100, 'sale': 150})`

In [ ]:
def bridge_revenue(orders, tags):
    """
    An order can have several tags. Return (revenue by naive join, true revenue,
    revenue per tag).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bridge_revenue, [
    (([(1, 100), (2, 50)], [(1, 'sale'), (1, 'gift'), (2, 'sale')]), (250, 150, {'gift': 100, 'sale': 150})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops over both lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary for amounts. Each tag's total is valid, but they don't add up to the true total.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops over both lists.

```python
def bridge_revenue(orders, tags):
    naive = 0
    per = {}
    for oid, amt in orders:
        for t_oid, tag in tags:
            if oid == t_oid:
                naive += amt
                per[tag] = per.get(tag, 0) + amt
    return (naive, sum(a for _, a in orders), dict(sorted(per.items())))
```

**Cost:** O(n·m) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary for amounts. Each tag's total is valid, but they don't add up to the true total.

```python
def bridge_revenue(orders, tags):
    amt = dict(orders)
    per = {}
    for oid, tag in tags:
        per[tag] = per.get(tag, 0) + amt[oid]
    return (sum(amt[o] for o, _ in tags), sum(amt.values()), dict(sorted(per.items())))
```

**Cost:** O(n + m) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops over both lists. That costs O(n·m) time and O(t) space."

"I can do better: A dictionary for amounts. Each tag's total is valid, but they don't add up to the true total. That's O(n + m) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/bridge-table-avoid-double-counting)

<a id="q20"></a>
### Q20. Late-Arriving Fact

🔴 Stretch · *Which dimension row applies?*

versions are (key, start) sorted by start. For each fact date, return the key active on that date, or −1 if before all of them.

**Examples**

- `versions = [(1, 10), (2, 20), (3, 30)], dates = [5, 10, 25, 99]` → `[-1, 1, 2, 3]`

In [ ]:
def active_key(versions, dates):
    """
    versions are (key, start) sorted by start. For each fact date, return the
    key active on that date, or −1 if before all of them.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(active_key, [
    (([(1, 10), (2, 20), (3, 30)], [5, 10, 25, 99]), [-1, 1, 2, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan versions for each fact.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Binary search on the start dates.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Scan versions for each fact.

```python
def active_key(versions, dates):
    out = []
    for d in dates:
        key = -1
        for k, start in versions:
            if start <= d:
                key = k
        out.append(key)
    return out
```

**Cost:** O(f·v) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Binary search on the start dates.

```python
def active_key(versions, dates):
    import bisect
    starts = [s for _, s in versions]
    return [versions[bisect.bisect_right(starts, d) - 1][0] if d >= starts[0] else -1 for d in dates]
```

**Cost:** O(f log v) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan versions for each fact. That costs O(f·v) time and O(f) space."

"I can do better: Binary search on the start dates. That's O(f log v) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/data-modeling/late-arriving-fact)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is grain?</b></summary>

What one row represents. Declaring it first prevents most modelling and reporting errors.

</details>

<details>
<summary><b>2. Fact vs dimension?</b></summary>

Facts record events and measures. Dimensions describe them with context used for filtering and grouping.

</details>

<details>
<summary><b>3. Star vs snowflake schema?</b></summary>

A star keeps dimensions denormalised, so joins are simple. A snowflake normalises dimensions into more tables, saving some space but adding joins.

</details>

<details>
<summary><b>4. Explain SCD types 1, 2 and 3.</b></summary>

Type 1 overwrites. Type 2 adds a new row with validity dates, keeping full history. Type 3 keeps limited history in an extra column.

</details>

<details>
<summary><b>5. Why use surrogate keys?</b></summary>

Natural keys can change or get reused. Surrogate keys are stable, and let one customer have many SCD 2 versions.

</details>

<details>
<summary><b>6. Normalise or denormalise?</b></summary>

Normalise for transactional systems, where consistent updates matter. Denormalise for analytics, where read speed and simplicity matter.

</details>

<details>
<summary><b>7. Scenario: design a model for an e-commerce store.</b></summary>

Start with fact\_order\_lines at one row per product per order, with dimensions for customer (SCD 2), product, date and store. Add a daily snapshot for inventory, as stock levels are semi-additive.

</details>

<details>
<summary><b>8. Scenario: an analyst says numbers doubled after joining a dimension. What's wrong?</b></summary>

The dimension isn't unique on the join key, often an SCD 2 table joined without date conditions. Check the grain, then add the point-in-time condition.

</details>

<details>
<summary><b>9. Scenario: a fact arrives for a customer not yet in the dimension. What do you do?</b></summary>

Don't drop it. Insert a placeholder dimension row, sometimes called an inferred member, or use an unknown key, and fix it up when the real record arrives.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Grain: the most important word in modelling
- [ ] I can explain: Star schemas: facts and dimensions
- [ ] I can explain: Slowly changing dimensions

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Unknown Dimension Member
- [ ] [Q17](#q17) Point-in-Time Join
- [ ] [Q18](#q18) SCD Type 2: Keep History
- [ ] [Q19](#q19) Bridge Table: Avoid Double Counting
- [ ] [Q20](#q20) Late-Arriving Fact

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.